In [1]:
from transformers import AutoTokenizer, AutoModel
from torch import nn
import torch
import torch.nn.functional as F
import os
import numpy as np
import random
import math
from tqdm import tqdm
import mteb
from mteb import ModelMeta # Import cho MTEB bản mới
from mteb.abstasks.AbsTaskRetrieval import AbsTaskRetrieval
from mteb.abstasks.TaskMetadata import TaskMetadata

seed=42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

/home/hungpv/miniconda3/envs/vllm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from types import SimpleNamespace


Arguments = SimpleNamespace

args = Arguments(
    dataset='mteb/fiqa',
    batch_size=32,
    val_batch_size=128,

    query_max_len=512,
    passage_max_len=512,

    temperature=0.02,

    student_device='cuda:0',
    num_train_epochs=5,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.05,

    output_dir='ckpt/fiqa/mini768-talas/bert-checkpoint',
    student_model='models/MiniLMv2-L6-H768-distilled-from-BERT-Base',
    student_tokenizer='bert-base-uncased',
)


teacher_embedding_path = 'data/fiqa/bge-m3.pt'

In [ ]:
def get_embedding(hidden_state, attention_mask):
    cls_token = hidden_state[:, 0:1, :]

    attention_scores = torch.bmm(cls_token, hidden_state.transpose(1, 2))
    attention_scores = attention_scores.squeeze(1)

    hidden_dim = hidden_state.size(-1)
    attention_scores = attention_scores / math.sqrt(hidden_dim)

    extended_attention_mask = (1.0 - attention_mask) * -1e9
    # extended_attention_mask[: 0] = -1e9
    attention_scores = attention_scores + extended_attention_mask
    weights = F.softmax(attention_scores, dim=-1)
    weights_expanded = weights.unsqueeze(-1)  # (batch_size, seq_len, 1)

    pooled_vector = torch.sum(hidden_state * weights_expanded, dim=1)

    return pooled_vector

In [4]:
from torch.nn.functional import pad
from torch.utils.data import Dataset
import pandas as pd
import torch
from transformers import PreTrainedTokenizer
from dataclasses import dataclass
import json
from datasets import load_dataset

from torch.utils.data import DataLoader, Dataset

class FiQATrainDataset(Dataset):
    def __init__(self, dataset, teacher_embedding_path):
        self.dataset = dataset
        qrels = load_dataset(dataset, "default")
        queries = load_dataset(dataset, "queries")
        corpus = load_dataset(dataset, "corpus")

        self.qrels = qrels["train"]

        query_data = queries["queries"]
        corpus_data = corpus["corpus"]
        

        self.query_map = {
            str(row["_id"]): row["text"]
            for row in query_data
        }

        # id -> document text
        self.corpus_map = {
            str(row["_id"]): row["text"]
            for row in corpus_data
        }

        self.teacher_embedding = torch.load(teacher_embedding_path)
        self.teacher_embedding_size = self.teacher_embedding["query_embedding"].size(-1)

    def __len__(self):
        return len(self.qrels)

    def __getitem__(self, idx):
        row = self.qrels[idx]

        query_id = str(row["query-id"])
        corpus_id = str(row["corpus-id"])

        query = self.query_map[query_id]
        corpus = self.corpus_map[corpus_id]

        teacher_embedding = {
            'query_embedding': self.teacher_embedding["query_embedding"][idx],
            'corpus_embedding': self.teacher_embedding["corpus_embedding"][idx]
        }

        return {'query': query, 'corpus': corpus}, teacher_embedding

@dataclass
class BiDataCollator:
    student_tokenizer: PreTrainedTokenizer = None
    max_len: int = 64
    return_tensors: str = 'pt'
    padding: bool = True


    def __call__(self, batch):
        query = [item[0]["query"] for item in batch]
        corpus = [item[0]["corpus"] for item in batch]

        teacher_embeddings = {
            'query_embedding': torch.stack([item[1]['query_embedding'] for item in batch]),
            'corpus_embedding': torch.stack([item[1]['corpus_embedding'] for item in batch])
        }

        query_inputs = self.student_tokenizer(
            query,
            truncation=True,
            padding=self.padding,
            max_length=self.max_len,
            return_tensors=self.return_tensors,
        )

        corpus_inputs = self.student_tokenizer(
            corpus,
            truncation=True,
            padding=self.padding,
            max_length=self.max_len,
            return_tensors=self.return_tensors,
        )

        return {'query': query_inputs, 
                'corpus': corpus_inputs}, teacher_embeddings

In [5]:
class FiQADev(AbsTaskRetrieval):

    metadata = TaskMetadata(
        name="FiQADev",
        description="FiQA retrieval development split",
        reference="https://sites.google.com/view/fiqa/",
        dataset={
            "path": "mteb/fiqa",
            "revision": "main",
        },
        type="Retrieval",
        category="s2p",
        modalities=["text"],
        eval_splits=["dev"],
        eval_langs=["eng-Latn"],
        main_score="ndcg_at_10",
        date=("2018-01-01", "2018-12-31"),
        domains=["Financial"],
        task_subtypes=[],
        license=None,
        annotations_creators="derived",
        dialect=[],
        text_creation="found",
        bibtex_citation="",
        n_samples={
            "dev": 1238,
        },

        avg_character_length=None,
    )

    def load_data(self, **kwargs):
        if self.data_loaded:
            return

        qrels = load_dataset(
            "mteb/fiqa",
            "default",
        )

        queries = load_dataset(
            "mteb/fiqa",
            "queries",
        )

        corpus = load_dataset(
            "mteb/fiqa",
            "corpus",
        )

        query_dict = {
            str(row["_id"]): row["text"]
            for row in queries["queries"]
        }

        corpus_dict = {
            str(row["_id"]): {
                "title": "",
                "text": row["text"],
            }
            for row in corpus["corpus"]
        }

        relevant_docs = {}

        for row in qrels["dev"]:
            qid = str(row["query-id"])
            cid = str(row["corpus-id"])
            score = int(row["score"])

            if qid not in relevant_docs:
                relevant_docs[qid] = {}

            relevant_docs[qid][cid] = score

        # ------------------------
        # Important:
        # MTEB 1.x indexes by split
        # ------------------------
        self.queries = {
            "dev": query_dict,
        }

        self.corpus = {
            "dev": corpus_dict,
        }

        self.relevant_docs = {
            "dev": relevant_docs,
        }

        self.data_loaded = True

In [6]:
student_tokenizer = AutoTokenizer.from_pretrained(args.student_tokenizer)

train_dataset = FiQATrainDataset('mteb/fiqa', teacher_embedding_path)

train_collate = BiDataCollator(student_tokenizer,
                               max_len = args.passage_max_len,
                               return_tensors = 'pt', padding = True)

train_loader = DataLoader(train_dataset, batch_size=args.batch_size,
                          shuffle=True, collate_fn=train_collate)

args.teacher_embedding_dimension = train_dataset.teacher_embedding_size

In [ ]:
class MTEBModel(nn.Module):
    def __init__(self, model, tokenizer, args):
        super().__init__()
        self.model = model
        self.tokenizer = tokenizer
        self.args = args
        self.device = torch.device(args.student_device)
        config = model.config
        self.model_name = f"custom/{config._name_or_path}"

        n_parameters = sum(
            p.numel()
            for p in model.parameters()
        )

        embed_dim = config.hidden_size

        max_tokens = getattr(
            config,
            "max_position_embeddings",
            args.query_max_len,
        )

        self.mteb_model_meta = ModelMeta(
            name=self.model_name,
            revision=getattr(config, "_commit_hash", None) or "main",

            n_parameters=n_parameters,
            embed_dim=embed_dim,
            max_tokens=max_tokens,

            languages=["eng-Latn"],
            similarity_fn_name="cosine",

            framework=["PyTorch"],

            release_date="2024-01-01",
            memory_usage_mb=None,
            license="apache-2.0",

            open_weights=True,
            public_training_code=None,
            public_training_data=True,

            use_instructions=False,
            training_datasets=None,
        )


    def get_embedding(self, hidden_state, attention_mask):
        cls_token = hidden_state[:, 0:1, :]

        attention_scores = torch.bmm(cls_token, hidden_state.transpose(1, 2))
        attention_scores = attention_scores.squeeze(1)

        hidden_dim = hidden_state.size(-1)
        attention_scores = attention_scores / math.sqrt(hidden_dim)

        extended_attention_mask = (1.0 - attention_mask) * -1e9
        # extended_attention_mask[: 0] = -1e9
        attention_scores = attention_scores + extended_attention_mask
        weights = F.softmax(attention_scores, dim=-1)
        weights_expanded = weights.unsqueeze(-1)  # (batch_size, seq_len, 1)

        pooled_vector = torch.sum(hidden_state * weights_expanded, dim=1)

        return pooled_vector

    @torch.no_grad()
    def encode(
        self,
        sentences,
        batch_size=64,
        **kwargs,
    ):
        all_embeddings = []

        for start in range(0, len(sentences), batch_size):
            batch_texts = sentences[start:start + batch_size]

            inputs = self.tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=self.args.passage_max_len,
                return_tensors="pt",
            )

            inputs = {
                k: v.to(self.device)
                for k, v in inputs.items()
            }

            outputs = self.model(**inputs, return_dict=True,)

            hidden_states = outputs.last_hidden_state

            # Dùng đúng attention pooling của bạn
            embeddings = self.get_embedding(
                hidden_state=hidden_states,
                attention_mask=inputs["attention_mask"],
            )

            # Normalize để retrieval bằng cosine/dot-product
            embeddings = F.normalize(embeddings, p=2, dim=-1,)

            all_embeddings.append(embeddings.cpu())

        return torch.cat(all_embeddings,dim=0).numpy()
        

In [8]:
class Trainer:
    def __init__(self, student, projectors, args):
        super().__init__()

        self.student = student.train()
        self.projector = projectors
        self.device = student.device
        self.n_layer = student.config.num_hidden_layers

        self.cross_entropy = nn.CrossEntropyLoss(reduction='mean')
        self.mse_loss = nn.MSELoss(reduction='mean')

        self.args = args

        self.step = 0

    def compute_scores(self, q_reps, p_reps, temperature=1.0):
        scores = torch.matmul(q_reps, p_reps.transpose(0, 1)) / temperature
        scores = scores.view(q_reps.size(0), -1)
        return scores

    def std_norm(self, embeddings):
        std_embeddings = embeddings.std(dim=-1, keepdim=True).detach()
        norm_embeddings = embeddings / std_embeddings
        return norm_embeddings

    def mse_dim_weight_loss(self, student_embeddings, teacher_embeddings, power=1.0):
        std_embeddings = teacher_embeddings.std(dim=(0))
        mean_std = std_embeddings.mean()
        std_scaled = std_embeddings / mean_std
        weights = std_scaled ** power

        squared_diff = (student_embeddings - teacher_embeddings) ** 2
        weighted_squared_diff = squared_diff * weights
        weighted_mse_loss = weighted_squared_diff.mean()

        return weighted_mse_loss

    def structure_loss(self, student_embeddings, teacher_embeddings):
        student_embeddings = F.normalize(student_embeddings, p=2, dim=-1)
        teacher_embeddings = F.normalize(teacher_embeddings, p=2, dim=-1)

        student_similarity = student_embeddings @ student_embeddings.transpose(-1, -2)
        teacher_similarity = teacher_embeddings @ teacher_embeddings.transpose(-1, -2)

        loss = F.mse_loss(student_similarity, teacher_similarity)

        return loss

    def cosine_loss(self, student_embeddings, teacher_embeddings):
        cos_sim = F.cosine_similarity(student_embeddings, teacher_embeddings, dim=-1)
        cos_sim_loss = 1 - cos_sim
        return cos_sim_loss.mean()

    def get_embedding(self, hidden_state, attention_mask):
        return get_embedding(hidden_state, attention_mask)

    def distill_loss(self, student_q_outputs, student_p_outputs, teacher_embedings,
                      query_attention_mask, passage_attention_mask):
 
        q_hidden_states = student_q_outputs.hidden_states
        p_hidden_states = student_p_outputs.hidden_states

        if q_hidden_states is None or p_hidden_states is None:
            return 0, 0

        teacher_q_embed = teacher_embedings['query_embedding']
        teacher_p_embed = teacher_embedings['corpus_embedding']
        teacher_embed_cat = torch.cat([teacher_q_embed, teacher_p_embed], dim=0)

        TAMD_loss, LASD_loss = 0, 0

        n_layer_distill = 2
        for i in range(self.n_layer - n_layer_distill + 1, self.n_layer + 1):
            h_embed_q = self.get_embedding(q_hidden_states[i], query_attention_mask)
            h_embed_p = self.get_embedding(p_hidden_states[i], passage_attention_mask)
            h_embed = torch.cat([h_embed_q, h_embed_p], dim=0)  # gộp batch q + p

            h_proj = self.projector[i - 1](h_embed)
            TAMD_loss += self.cosine_loss(h_proj, teacher_embed_cat)

        TAMD_loss = TAMD_loss / n_layer_distill

        for i in range(1, self.n_layer):
            h_embed_q = self.get_embedding(q_hidden_states[i], query_attention_mask)
            h_embed_p = self.get_embedding(p_hidden_states[i], passage_attention_mask)
            h_embedding = torch.cat([h_embed_q, h_embed_p], dim=0)

            next_h_embed_q = self.get_embedding(q_hidden_states[i + 1], query_attention_mask)
            next_h_embed_p = self.get_embedding(p_hidden_states[i + 1], passage_attention_mask)
            next_h_embedding = torch.cat([next_h_embed_q, next_h_embed_p], dim=0)

            LASD_loss += self.structure_loss(h_embedding, next_h_embedding)

        LASD_loss = LASD_loss / (self.n_layer - 1)

        return TAMD_loss, LASD_loss

    def compute_InfoNCE_loss(self, scores, target):
        return self.cross_entropy(scores, target)

    def compute_loss(self, student_inputs, teacher_embedings=None):
        query_inputs = student_inputs['query']
        passage_inputs = student_inputs['corpus']

        query_inputs = {key: value.to(self.device) for key, value in query_inputs.items()}
        passage_inputs = {key: value.to(self.device) for key, value in passage_inputs.items()}

        student_q_outputs = self.student(**query_inputs, return_dict=True)
        student_p_outputs = self.student(**passage_inputs, return_dict=True)

        student_q_embeddings = self.get_embedding(student_q_outputs.last_hidden_state,
                                                   query_inputs['attention_mask'])
        student_p_embeddings = self.get_embedding(student_p_outputs.last_hidden_state,
                                                   passage_inputs['attention_mask'])
        q_dense_vecs = nn.functional.normalize(student_q_embeddings, dim=-1)
        p_dense_vecs = nn.functional.normalize(student_p_embeddings, dim=-1)

        targets = torch.arange(q_dense_vecs.size(0), device=self.device, dtype=torch.long)

        # dense loss
        dense_scores = self.compute_scores(q_dense_vecs, p_dense_vecs, self.args.temperature)
        simcse_loss = self.compute_InfoNCE_loss(dense_scores, targets)

        TAMD_loss, LASD_loss = self.distill_loss(
            student_q_outputs,
            student_p_outputs,
            teacher_embedings,
            query_inputs['attention_mask'],
            passage_inputs['attention_mask'],
        )

        loss = simcse_loss + TAMD_loss + LASD_loss

        self.step += 1

        return loss, simcse_loss

In [9]:
from torch import optim
# from muon import Muon
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
from transformers import get_scheduler

In [10]:
task = FiQADev()

evaluation = mteb.MTEB(
    tasks=[task]
)

In [11]:
model = AutoModel.from_pretrained(args.student_model, device_map=args.student_device,
                                  output_hidden_states=True,
                                  # output_attentions
                                 )

mteb_model = MTEBModel(model, student_tokenizer, args)

proj_hidden_layers = nn.ModuleList()

for _ in range(model.config.num_hidden_layers):
    proj = nn.Linear(model.config.hidden_size, args.teacher_embedding_dimension, bias=False)
    with torch.no_grad():
        proj.weight.normal_(mean=0.0, std=1e-3)
    proj_hidden_layers.append(proj)

proj_hidden_layers = proj_hidden_layers.to(model.device)

trainer = Trainer(model, proj_hidden_layers, args)

In [12]:
muon_params = []
adamw_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue

    is_edge_or_head = any(keyword in name
                          for keyword in ["embeddings", "pooler", "projector",
                                          "classifier", "head", "layer.0",
                                          "layer.1", "layer.2", "layer.3", "layer.11"])

    is_1d_or_norm = param.ndim < 2 or "LayerNorm" in name or "bias" in name

    if not is_edge_or_head and not is_1d_or_norm:
        muon_params.append(param)
    else:
        adamw_params.append(param)


# opt_muon = optim.Muon(muon_params, lr=args.learning_rate,
#                       momentum=0.95, adjust_lr_fn="match_rms_adamw")

opt_muon = optim.Muon(muon_params, lr=0.001, momentum=0.95)

opt_adamw = optim.AdamW(adamw_params, lr=args.learning_rate)
opt_adamw.add_param_group({"params": proj_hidden_layers.parameters(),
                           "lr": 1e-4, "weight_decay": 0.01})


In [ ]:
num_steps = len(train_loader)
total_training_steps = num_steps * args.num_train_epochs

scheduler_adamw = get_scheduler(
    name='cosine_with_min_lr',
    optimizer=opt_adamw,
    num_warmup_steps=int(total_training_steps * args.warmup_ratio),
    num_training_steps=total_training_steps,
    scheduler_specific_kwargs={'min_lr': 5e-6}
)

scheduler_muon = get_scheduler(
    name='cosine_with_min_lr',
    optimizer=opt_muon,
    num_warmup_steps=int(total_training_steps * args.warmup_ratio),
    num_training_steps=total_training_steps,
    scheduler_specific_kwargs={'min_lr': 5e-6}
)

best_result = 0

# Training loop
for epoch in range(args.num_train_epochs):
    print(('\n' + '%8s' + '%14s' + '%17s' * 2) % ('epoch', 'memory', 'loss', 'student_loss'))
    p_bar = tqdm(train_loader, total=len(train_loader))
    loss_total = 0
    student_loss_total = 0
    step = 0
    model.train()
    for batch in p_bar:
        student_inputs, teacher_embeddings = batch
        teacher_embeddings = {key: value.to(model.device) for key, value in teacher_embeddings.items()}

        opt_muon.zero_grad(set_to_none=True)
        opt_adamw.zero_grad(set_to_none=True)

        with autocast(dtype=torch.bfloat16):
            loss, student_loss = trainer.compute_loss(student_inputs, teacher_embeddings)

        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(trainer.student.parameters(), max_norm=1.0)

        # Optimizer step
        opt_muon.step()
        opt_adamw.step()

        # Update cả 2 scheduler
        scheduler_adamw.step()
        scheduler_muon.step()

        loss_total += loss.item()
        student_loss_total += student_loss.item()
        step += 1

        memory = f'{torch.cuda.memory_reserved() / 1E9:.4g}G'
        s = ('%8s' + '%14s' + '%17.5g' * 2) % (f'{epoch + 1}/{args.num_train_epochs}', memory,
                                                loss_total / step, student_loss_total / step)
        p_bar.set_description(s)

        if torch.isnan(loss):
            print("CẢNH BÁO: Loss bị NaN, ngắt epoch hiện tại!")
            break

    trainer.student.save_pretrained(args.output_dir + f'-epoch{epoch}')


   epoch        memory             loss     student_loss


  0%|          | 0/443 [00:00<?, ?it/s]/tmp/ipykernel_2388596/2924333316.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.bfloat16):
     1/5        5.742G            1.584           1.3335: 100%|██████████| 443/443 [01:53<00:00,  3.92it/s]



   epoch        memory             loss     student_loss


     2/5        5.746G          0.67099          0.49405: 100%|██████████| 443/443 [01:54<00:00,  3.86it/s]



   epoch        memory             loss     student_loss


     3/5        5.746G          0.41071          0.24455: 100%|██████████| 443/443 [01:54<00:00,  3.86it/s]



   epoch        memory             loss     student_loss


     4/5        5.746G          0.29464          0.13591: 100%|██████████| 443/443 [01:54<00:00,  3.85it/s]



   epoch        memory             loss     student_loss


     5/5        5.746G          0.25038         0.097156: 100%|██████████| 443/443 [02:01<00:00,  3.65it/s]


In [ ]:
eval_tasks = mteb.get_tasks(
    tasks=["ArguAna","SciFact","FiQA2018"]
)

model = AutoModel.from_pretrained("ckpt/fiqa/mini768-talas/bert-checkpoint-epoch4", 
                                  device_map=args.student_device,
                                  output_hidden_states=True)

model.eval()

mteb_model = MTEBModel(model, student_tokenizer, args)

with autocast(dtype=torch.bfloat16):
    evaluation = mteb.MTEB(tasks=eval_tasks)
    results = evaluation.run(
        mteb_model,
        output_folder=args.output_dir,
        overwrite_results=True,
        eval_splits=["test"],
        encode_kwargs={
            "batch_size": args.val_batch_size,
        },
    )

print("Kết quả đánh giá cuối cùng:")
for res in results:
    print(res.task_name)
    print(res.scores['test'][0]['main_score'])

/tmp/ipykernel_2393913/1835395020.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.bfloat16):


───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Retrieval

- ArguAna, s2p

- SciFact, s2p

- FiQA2018, s2p

Kết quả đánh giá cuối cùng:
ArguAna
0.43912
SciFact
0.32765
FiQA2018
0.25275
